# ActionSearchTool — 단독 실행과 근거 검증 (11/05 B01~B06, B09)

`src/rag/school/action_tool.py`의 `action_search(tool_input, deps) -> ToolResult`를 **실제 공유 Neo4j**로 실행한다.

- 위에서부터 실행. 이 노트북은 DB에 쓰지 않는다(읽기만).
- 단위 검사(가짜 드라이버)는 `tests/rag/school/test_school_action_tool.py`. 여기는 **실DB 증빙**이다.
- 계약 타입·검사는 `contract_v2_local.py`(공통 계약 v2.0 §9 원문 사본). 유종혁 공통본이 오면 import만 교체.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

from index.school.dictionaries import load_dictionaries
from rag.school.graph import default_deps

deps = default_deps()            # driver·database·embedder. driver는 이 노트북이 소유하고 마지막 셀에서 닫는다.
deps["books"] = load_dictionaries()
DRIVER, DATABASE = deps["driver"], deps["database"]
ALLOWED = ["moe_school_manual_v3", "gyeonggi_school_guide_2025", "kdca_measles_2026",
           "kdca_diagnosis_reporting_2026", "moleg_infectious_disease_act_21521"]
print("Neo4j 접속 확인(접속 정보는 출력하지 않음) / embedder:", type(deps["embedder"]).__name__)

Neo4j 접속 확인(접속 정보는 출력하지 않음) / embedder: OpenAIEmbeddings


## B01. DB 기능·스키마 점검 (읽기 전용)

문서에 v3.1이 적혀 있다는 사실이 아니라 실DB 속성 채움으로 검색 경로를 고른다. 선정쪽수 밖 청크(경기도 54·56쪽, 11/05 결정으로 DB에 유지)는 Tool이 쓰지 않는다.

In [2]:
from rag.school.capabilities import inspect_search_capabilities

cap = inspect_search_capabilities(DRIVER, DATABASE)
deps["capabilities"] = cap
print("확인 시각:", cap["checked_at"])
display(pd.DataFrame(cap["documents"]))
display(pd.DataFrame(cap["chunks"]))
display(pd.DataFrame(cap["actions"]))
print("DONE_BY basis:", cap["done_by_basis"])
display(pd.DataFrame(cap["indexes"])[["name", "type", "state", "properties"]])
print("사용 가능 경로:", cap["paths"])
print("정책:", cap["policy"])
print("선정쪽수 밖 청크(쓰지 않음):", cap["chunks_outside_active_pages"])

확인 시각: 2026-11-05T03:59:32+00:00


,document_id,title,version,region,chunks
0,gyeonggi_school_guide_2025,경기도 학교 감염병 예방관리 실무 가이드북,2024.05 발행본(판권면),GYEONGGI,814
1,moe_school_manual_v3,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초·중·고·특수학교용),제3차 개정판,NATIONAL,325


,document_id,chunks,embedded,min_page,max_page
0,gyeonggi_school_guide_2025,814,814,54,396
1,moe_school_manual_v3,325,325,36,226


,document_id,actions,with_stage,stage_unspecified,with_step_order,with_done_by,with_concept,with_disease
0,gyeonggi_school_guide_2025,291,291,27,129,152,176,91
1,moe_school_manual_v3,179,179,54,109,161,111,6


DONE_BY basis: [{'basis': 'EXAMPLE', 'n': 340}]


,name,type,state,properties
0,action_scope,RANGE,ONLINE,[scope]
1,action_stage,RANGE,ONLINE,[stage]
2,chunk_embedding,VECTOR,ONLINE,[embedding]
3,chunk_text,FULLTEXT,ONLINE,[text]


사용 가능 경로: {'fixed_cypher_role_stage': True, 'fulltext': True, 'vector': True, 'v31_stage': True, 'v31_basis': True}
정책: 역할·단계가 질문에 있으면 고정 Cypher(DONE_BY·stage)로 먼저 찾고, 질문 표현은 청크 검색(벡터가 준비되면 hybrid, 아니면 fulltext)으로 보완한다. stage가 비었거나 UNSPECIFIED인 행동은 단계 주장을 하지 않는다. basis가 없는 DONE_BY는 담당 근거로 쓰지 않는다.
선정쪽수 밖 청크(쓰지 않음): [('gyeonggi_school_guide_2025', 54, 3), ('gyeonggi_school_guide_2025', 56, 5)]


## T02 유형: 병명 없는 학교 행동 질문 → ToolResult

ToolInput은 메인 그래프가 만드는 것과 같은 모양이다. `task`는 필수다.

In [3]:
from rag.school.action_tool import action_search
from rag.school.contract_v2_local import validate_tool_result

deps["trace"] = []
tool_input = {"question": "학교에서 유증상 학생이 발견되면 담임교사와 보건교사는 각각 무엇을 확인하나요?",
              "task": "학교가 수행하는 행동, 역할, 단계 및 근거 찾기", "region": "GYEONGGI",
              "allowed_document_ids": ALLOWED}
result = action_search(tool_input, deps)
validate_tool_result(result, tool_input, expected_tool="action_search")
print("status:", result["status"], "/ Finding", len(result["findings"]), "/ Citation", len(result["citations"]),
      "/ GraphPath", len(result["graph_paths"]))
for w in result["warnings"]:
    print("warning:", w)
display(pd.DataFrame([{"claim": f["claim"], "chunk_ids": ", ".join(c.split(":", 1)[1] for c in f["chunk_ids"]),
                       "action": ", ".join(a.split(":", 1)[1] for a in f["action_ids"]),
                       "concept": ", ".join(f["concept_ids"])} for f in result["findings"]]))

status: OK / Finding 12 / Citation 11 / GraphPath 8


,claim,chunk_ids,action,concept
0,[교육부 매뉴얼 PDF 54쪽 · 원문 배치: 대응 제1단계] 원문: “ 감염병 (의심)환자가 교내에서 혼자 이동하는 것은 원칙적으로 금지  이동 시에는 담임(담당) 교사가 동행하며 2m 정도의 거리 유지...,p0054:b01,,
1,[교육부 매뉴얼 PDF 197쪽] 원문: “7) 교사 및 학부모의 역할 ∙ 정기적인 예방접종 권장 ∙ 보건(담당)교사 및 담임교사는 학교 내 (의심)환자 발생에 따라 교육(지원)청에 보고 및 조치함”,p0197:b02,,
2,"[교육부 매뉴얼 PDF 57쪽 · 원문 배치: 대응 제1단계] 원문: “ 보호자에게 의료기관 진료 여부를 확인하고 미진료 시 진료를 받도록 하며, 이때 출결 관련 필요 서류 및 학교 홈페이지의 관련 서식 이...",p0057:b01,,
3,[교육부 매뉴얼 PDF 207쪽] 원문: “∙ 보건(담당)교사 및 담임교사는 학교 내 (의심)환자 발생에 따라 보고 및 조치함 ∙ 가정과 학교에서 노출가능성이 있는 사람들에게 증상 감시를 권고함 ∙ 성홍열 의...,p0207:b02,,
4,[경기도 가이드북 PDF 166쪽 · 대응 제1단계] 원문이 제시한 담당 예시: 담임교사 — 감염병 유증상자를 발견한다.,p0166:t01r01,p0166:a01,DETECT_SYMPTOMATIC
5,"[교육부 매뉴얼 PDF 36쪽, 모의훈련 시나리오 예시 · 대응 제1단계] 원문이 제시한 담당 예시: 보건교사 — 환자의 증상과 징후를 관찰하고 감염병이 의심되는지를 확인한다.",p0036:t01r02,p0036:a14,DETECT_SYMPTOMATIC
6,[경기도 가이드북 PDF 167쪽 · 대응 제1단계] 원문이 제시한 담당 예시: 보건교사 — 감염병을 의심할 수 있는 증상인지 확인한다.,p0167:t01r03,p0167:a28,DETECT_SYMPTOMATIC
7,"[교육부 매뉴얼 PDF 43쪽, 모의훈련 시나리오 예시 · 단계 미기재] 원문이 제시한 담당 예시: 담임교사 — 결석·조퇴·지각 사유를 확인하고 의심환자를 파악한다.",p0043:t01r02,p0043:a19,DETECT_SYMPTOMATIC
8,[경기도 가이드북 PDF 164쪽 · 대응 제1단계] 원문에 담당 미기재 — 감염병(의심) 환자 발생 여부를 확인한다. (조건: 결과를 확인 후),p0164:t01r05,p0164:a09,DETECT_SYMPTOMATIC
9,[교육부 매뉴얼 PDF 56쪽 · 대응 제1단계] 원문에 담당 미기재 — 학생을 최대한 안정시킨 상태에서 증상 변화를 주의 깊게 관찰한다.,p0056:t01r04,p0056:a13,


### 실행 기록: 계획 → 조회 경로(고정 Cypher·청크 검색·행동 전수) → 근거 재확인 → DB 대조

In [4]:
t = deps["trace"][-1]
print("계획:", t["plan"])
display(pd.DataFrame(t["log"]))
display(pd.DataFrame(t["selected"]))
print("DB 대조:", t["db_report"])

계획: {'roles': ['HOMEROOM_TEACHER', 'HEALTH_TEACHER'], 'recipients': [], 'stages': ['RESPONSE_1'], 'diseases': [], 'mode': 'hybrid', 'broad': True, 'unresolved': []}


,step,roles,stages,rows,kept,hits,in_scope,linked_actions,candidates,selected,with_evidence
0,role_cypher,"[HOMEROOM_TEACHER, HEALTH_TEACHER]",[RESPONSE_1],74.0,54.0,NaN,NaN,NaN,NaN,NaN,NaN
1,chunk_hybrid,NaN,NaN,NaN,39.0,12.0,11.0,39.0,NaN,NaN,NaN
2,action_scan,NaN,NaN,470.0,438.0,NaN,NaN,NaN,NaN,NaN,NaN
3,evidence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,438.0,8.0,8.0
4,quotes,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,action_id,score,via
0,gyeonggi_school_guide_2025:p0166:a01,21.0,"[chunk, role, scan]"
1,moe_school_manual_v3:p0036:a14,15.5,"[role, scan]"
2,gyeonggi_school_guide_2025:p0167:a28,14.0,"[role, scan]"
3,moe_school_manual_v3:p0043:a19,11.0,[scan]
4,gyeonggi_school_guide_2025:p0164:a09,8.5,[scan]
5,moe_school_manual_v3:p0056:a13,6.5,[scan]
6,gyeonggi_school_guide_2025:p0166:a11,16.5,"[chunk, role, scan]"
7,moe_school_manual_v3:p0042:a02,14.0,"[chunk, scan]"


DB 대조: {'bad_citations': {}, 'bad_findings': {}, 'bad_paths': {}}


### 실제 Cypher(역할 직접 조회)와 Citation·GraphPath

In [5]:
import inspect
from rag.school import search

src = inspect.getsource(search.get_role_actions)
print(src[src.index('MATCH'):src.index('""", database')])
display(pd.DataFrame(result["citations"]))
print(json.dumps(result["graph_paths"][0], ensure_ascii=False, indent=1))

MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk)<-[:SUPPORTED_BY]-(a:SchoolAction)-[:DONE_BY]->(r:SchoolRole)
WHERE r.role_id IN $role_ids AND d.document_id IN $allowed AND ($stages IS NULL OR a.stage IN $stages)
WITH DISTINCT a, c, d
RETURN {ACTION_FIELDS}
ORDER BY document_id, chunk_id, a.step_order, action_id LIMIT $limit


,document_id,chunk_id,pdf_page,printed_page,excerpt
0,moe_school_manual_v3,moe_school_manual_v3:p0054:b01,54,40, 감염병 (의심)환자가 교내에서 혼자 이동하는 것은 원칙적으로 금지\n 이동 시에는 담임(담당) 교사가 동행하며 2m 정도의 거리 유지\n 담당 교사는 담임교사 권장 (만약 임신부 등 감염병 고위험군인 ...
1,moe_school_manual_v3,moe_school_manual_v3:p0197:b02,197,183,7) 교사 및 학부모의 역할\n∙ 정기적인 예방접종 권장\n∙ 보건(담당)교사 및 담임교사는 학교 내 (의심)환자 발생에 따라 교육(지원)청에 보고 및 조치함
2,moe_school_manual_v3,moe_school_manual_v3:p0057:b01,57,43," 보호자에게 의료기관 진료 여부를 확인하고 미진료 시 진료를 받도록 하며, 이때 출결 관련 필요 서류 및 학교\n홈페이지의 관련 서식 이용 방법 안내\n 감염병 증상으로 결석한 사실을 인지한 즉시 보건(담..."
3,moe_school_manual_v3,moe_school_manual_v3:p0207:b02,207,193,∙ 보건(담당)교사 및 담임교사는 학교 내 (의심)환자 발생에 따라 보고 및 조치함\n∙ 가정과 학교에서 노출가능성이 있는 사람들에게 증상 감시를 권고함\n∙ 성홍열 의심증상자는 즉시 의료기관으로 진료를 권고함
4,gyeonggi_school_guide_2025,gyeonggi_school_guide_2025:p0166:t01r01,166,156,담임교사|대응<br>제1단계|1. 감염병 유증상자 발견<br> - 보건교사에게 연락(마스크 착용 필요 여부 확인)<br>2. 보건실로 유증상자 이동<br> - 감염병(의심) 환자가 교내에서 혼자 이동하는 것은...
5,moe_school_manual_v3,moe_school_manual_v3:p0036:t01r02,36,22,"보건(담당)교사 | 1) 교사와 진한이가 보건실에 도착한 즉시 손을 씻도록 요청 2) 보건교사는 마스크를 착용하고, 체온 측정과 더불어 환자의 증상과 징후를 관찰한 후 감염병이 의심되는지를 확인 3) 진한이가..."
6,gyeonggi_school_guide_2025,gyeonggi_school_guide_2025:p0167:t01r03,167,157,보건교사|대응<br>제1단계|1. 감염병 여부 확인<br> - 감염병을 의심할 수 있는 증상인지를 확인하고 의료기관 진료 여부를 물어봄<br> - 발열을 호소하는 경우 체온을 측정하여 실제 발열 여부를 다시 ...
7,moe_school_manual_v3,moe_school_manual_v3:p0043:t01r02,43,29,"담임 교사 | 1) 보건교사의 유증상자 조사요령에 따라 본인의 반 학생들의 결석/조퇴/지각 사유 확인과 (의심)환자 파악 - 병원이송 및 결석, 조퇴학생의 인적사항(성명, 주소, 연락처, 입원병원명, 형제자매..."
8,gyeonggi_school_guide_2025,gyeonggi_school_guide_2025:p0164:t01r05,164,154,대응 제1단계 | 주요 내용 | 의료기관에 진료를 의뢰하여 결과를 확인 후 감염병(의심) 환자 발생 여부 확인
9,moe_school_manual_v3,moe_school_manual_v3:p0056:t01r04,56,42,방법 | 관찰실 내에서 교사와 학생은 2m 이상의 거리 유지; 유증상 학생이 있는 경우에는 복도 쪽 창문은 닫고 실외 쪽 창문은 모두 개방; 최대한 학생을 안정시킨 상태에서 주의 깊게 증상 변화 관찰; 필요시...


{
 "edges": [
  {
   "source_ref": "SchoolAction:gyeonggi_school_guide_2025:p0166:a01",
   "relationship_type": "SUPPORTED_BY",
   "target_ref": "Chunk:gyeonggi_school_guide_2025:p0166:t01r01"
  },
  {
   "source_ref": "Document:gyeonggi_school_guide_2025",
   "relationship_type": "HAS_CHUNK",
   "target_ref": "Chunk:gyeonggi_school_guide_2025:p0166:t01r01"
  },
  {
   "source_ref": "SchoolAction:gyeonggi_school_guide_2025:p0166:a01",
   "relationship_type": "DONE_BY",
   "target_ref": "SchoolRole:HOMEROOM_TEACHER"
  },
  {
   "source_ref": "SchoolAction:gyeonggi_school_guide_2025:p0166:a01",
   "relationship_type": "INSTANCE_OF",
   "target_ref": "ActionConcept:DETECT_SYMPTOMATIC"
  }
 ]
}


## B06. 실DB 진위 검사 — T08(가짜 ID)·T09(가짜·뒤집힌·허용 밖 경로)

조작한 결과가 검사를 통과하지 못해야 한다. 실패한 주장·경로는 제거되고 사유가 남는다(새 표현으로 고쳐 통과시키지 않음).

In [6]:
import copy
from rag.school.action_tool import check_graph_paths, verify_tool_result_db

forged = copy.deepcopy(result)
fake_chunk = "moe_school_manual_v3:p0036:t09r09"
forged["findings"][0]["chunk_ids"] = [fake_chunk]
forged["citations"].append({"document_id": "moe_school_manual_v3", "chunk_id": fake_chunk, "pdf_page": 36,
                            "printed_page": 22, "excerpt": "지어낸 문장"})
act_index = next(i for i, f in enumerate(forged["findings"]) if f["action_ids"])
forged["findings"][act_index]["action_ids"] = ["moe_school_manual_v3:p0036:a99"]
cleaned, report = verify_tool_result_db(forged, tool_input, DRIVER, DATABASE)
print("T08 가짜 청크:", report["bad_citations"])
print("T08 거절된 Finding:", report["bad_findings"])
print("검사 후 status:", cleaned["status"], "/ Finding", len(result["findings"]), "→", len(cleaned["findings"]))
assert fake_chunk in report["bad_citations"] and act_index in report["bad_findings"]
assert all(fake_chunk not in f["chunk_ids"] for f in cleaned["findings"])

edge = result["graph_paths"][0]["edges"][0]
reversed_edge = {"source_ref": edge["target_ref"], "relationship_type": edge["relationship_type"],
                 "target_ref": edge["source_ref"]}
made_up = {**edge, "relationship_type": "DONE_BY"}            # SUPPORTED_BY 자리에 없는 관계
paths = [result["graph_paths"][0], {"edges": [reversed_edge]}, {"edges": [made_up]}]
problems = check_graph_paths(paths, ALLOWED, DRIVER, DATABASE)
print("T09:", json.dumps(problems, ensure_ascii=False, indent=1))
assert 0 not in problems and 1 in problems and 2 in problems
gyeonggi_only = check_graph_paths([p for p in result["graph_paths"]
                                   if "moe_school_manual_v3" in json.dumps(p)][:1], ["gyeonggi_school_guide_2025"],
                                  DRIVER, DATABASE)
print("허용 밖 경로:", gyeonggi_only)
assert gyeonggi_only

T08 가짜 청크: {'moe_school_manual_v3:p0036:t09r09': ['DB에 없는 chunk_id']}
T08 거절된 Finding: {0: ['인용 청크 문제: moe_school_manual_v3:p0036:t09r09'], 4: ['DB에 없는 action_id: moe_school_manual_v3:p0036:a99']}
검사 후 status: PARTIAL / Finding 12 → 10


T09: {
 "1": [
  "방향이 반대인 관계: {'source_ref': 'Chunk:gyeonggi_school_guide_2025:p0166:t01r01', 'relationship_type': 'SUPPORTED_BY', 'target_ref': 'SchoolAction:gyeonggi_school_guide_2025:p0166:a01'}"
 ],
 "2": [
  "DB에 없는 관계: {'source_ref': 'SchoolAction:gyeonggi_school_guide_2025:p0166:a01', 'relationship_type': 'DONE_BY', 'target_ref': 'Chunk:gyeonggi_school_guide_2025:p0166:t01r01'}"
 ]
}


허용 밖 경로: {0: ['허용 문서 밖 경로: SchoolAction:moe_school_manual_v3:p0036:a14', '허용 문서 밖 경로: Chunk:moe_school_manual_v3:p0036:t01r02', '허용 문서 밖 경로: Document:moe_school_manual_v3']}


## 허용 범위: allowed=[] · preferred⊄allowed · 한 문서만 허용

In [7]:
empty = action_search({**tool_input, "allowed_document_ids": []}, deps)
print("allowed=[]:", empty["status"], empty["warnings"][:1])
assert empty["status"] == "NO_EVIDENCE" and not empty["citations"]

bad_pref = action_search({**tool_input, "preferred_document_ids": ["kdca_measles_2026", "not_allowed_doc"]}, deps)
print("preferred⊄allowed:", bad_pref["status"], bad_pref["warnings"])
assert bad_pref["status"] == "ERROR"

only_gg = action_search({**tool_input, "allowed_document_ids": ["gyeonggi_school_guide_2025"]}, deps)
docs = {c["document_id"] for c in only_gg["citations"]}
refs = {e[k].split(":", 1)[1].split(":")[0] for p in only_gg["graph_paths"] for e in p["edges"]
        for k in ("source_ref", "target_ref") if e[k].split(":", 1)[0] in ("Chunk", "Document", "SchoolAction")}
pages = sorted({c["pdf_page"] for c in only_gg["citations"]})
print("경기도만 허용:", only_gg["status"], docs, "/ 경로 문서:", refs, "/ 인용 쪽:", pages)
assert docs <= {"gyeonggi_school_guide_2025"} and refs <= {"gyeonggi_school_guide_2025"}
assert not {54, 56} & set(pages)            # 선정쪽수 밖 청크는 쓰지 않음

allowed=[]: NO_EVIDENCE ['허용 문서 목록이 비어 있어 검색하지 않았다.']
preferred⊄allowed: ERROR ['ToolInput 계약 위반: preferred docs are outside allowed docs']


경기도만 허용: OK {'gyeonggi_school_guide_2025'} / 경로 문서: {'gyeonggi_school_guide_2025'} / 인용 쪽: [164, 166, 167, 191, 196, 198, 204]


## 질문셋 v1.1 채점 결과 (Tool 단독)

채점은 `notebooks/school/평가결과/golden_eval_tool.py`로 돌리고(정답 청크는 채점에만 사용), 여기서는 저장 결과를 읽는다. 해석은 `평가결과/평가_20261105_기준선_v1.md`.

In [8]:
ev = json.loads((ROOT / "data/interim/school/eval_tool_final.json").read_text(encoding="utf-8"))
rows = pd.DataFrame(ev["rows"])
rows["정답 인용"] = rows["hit"].astype(str) + "/" + rows["total"].astype(str)
display(rows[["id", "split", "route", "status", "정답 인용", "n_find", "n_act", "paths", "sec"]])
print("정답 1개 이상 인용:", int((rows["hit"] > 0).sum()), "/", len(rows),
      "| 전부:", int((rows["hit"] == rows["total"]).sum()), "/", len(rows))

,id,split,route,status,정답 인용,n_find,n_act,paths,sec
0,S-Q01,dev,action_search,OK,3/5,12,8,8,3.8
1,S-Q02,dev,action_search,OK,1/4,12,8,8,2.8
2,S-Q03,eval,legal_search,PARTIAL,1/2,12,8,8,2.9
3,S-Q04,dev,crossdoc_graph_search,PARTIAL,2/3,12,8,8,2.5
4,S-Q05,dev,action_search,OK,2/2,12,8,8,2.5
5,S-Q06,eval,action_search,OK,0/1,8,8,8,2.6
6,S-Q07,dev,action_search,OK,2/2,12,8,8,3.3
7,S-Q08,dev,action_search,OK,2/2,12,8,8,2.8
8,S-Q09,eval,action_search,OK,1/1,12,8,8,2.7
9,S-Q10,dev,action_search,OK,0/3,12,8,8,2.7


정답 1개 이상 인용: 25 / 34 | 전부: 15 / 34


In [9]:
DRIVER.close()